# Pipeline Troubleshooting & EDA

Standalone notebook to step through the Gold layer transformation pipeline, verify aggregation logic, and explore alternative modeling approaches.

**Purpose:** Pinpoint issues in model/aggregation steps by reproducing Gold layer logic with full visibility into intermediate DataFrames.

**Scope:** Current Stores (7 states) features + actual sales for training → Predict sales for MA candidates.

**Does NOT include:** Partner POI details, viz layer prep, Genie space, or anything frontend-specific.

## 0. Setup & Parameters

In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
import xgboost as xgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr, explode, lit, when, coalesce
from pyspark.sql.window import Window

# ============================================================================
# PARAMETERS — defaults from databricks.yml
# ============================================================================
dbutils.widgets.text("catalog", "ioc_sandbox")
dbutils.widgets.text("bronze_schema", "geo_bronze")
dbutils.widgets.text("silver_schema", "geo_silver")
dbutils.widgets.text("gold_schema", "geo_gold")
dbutils.widgets.text("state_filter", "MA,MI,VA,NY,WA,MD,NJ")
dbutils.widgets.text("expansion_state", "MA")
dbutils.widgets.text("lce_locations_raw_table", "ioc_sandbox.ai_strategy.lce_locations_raw")
dbutils.widgets.text("carto_table", "dbmp_carto_spatial_features_usa_h3_res_8.carto.derived_spatialfeatures_usa_h3res8_v1_yearly_v3")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")
state_filter = dbutils.widgets.get("state_filter")
expansion_state = dbutils.widgets.get("expansion_state")
lce_locations_raw_table = dbutils.widgets.get("lce_locations_raw_table")
carto_table = dbutils.widgets.get("carto_table")

H3_RESOLUTION = 8

# Table references (derived from schema parameters)
CURRENT_STORES_TABLE   = f"{catalog}.{bronze_schema}.current_stores_raw"
ISOCHRONES_LCE_TABLE   = f"{catalog}.{silver_schema}.isochrones_lce"
CANDIDATE_ISO_TABLE    = f"{catalog}.{silver_schema}.candidate_isochrones"
H3_FEATURES_TABLE      = f"{catalog}.{silver_schema}.h3_features_clean"
COMPETITORS_TABLE      = f"{catalog}.{silver_schema}.pois_competitors"
PARTNERS_TABLE         = f"{catalog}.{silver_schema}.pois_partners"

# Pipeline gold outputs (for comparison)
PIPELINE_AGG_TABLE     = f"{catalog}.{gold_schema}.current_stores_features_agg"
PIPELINE_CAND_AGG      = f"{catalog}.{gold_schema}.candidates_features_agg"
PIPELINE_CAND_FINAL    = f"{catalog}.{gold_schema}.candidates_finalized"

print("Parameters (from databricks.yml):")
print(f"  catalog:          {catalog}")
print(f"  state_filter:     {state_filter}")
print(f"  expansion_state:  {expansion_state}")
print(f"  lce_locations:    {lce_locations_raw_table}")
print(f"  carto_table:      {carto_table}")
print(f"\nTable references:")
for name, table in [
    ("Current Stores", CURRENT_STORES_TABLE),
    ("Isochrones LCE", ISOCHRONES_LCE_TABLE),
    ("Candidate Isochrones", CANDIDATE_ISO_TABLE),
    ("H3 Features", H3_FEATURES_TABLE),
    ("Competitors", COMPETITORS_TABLE),
    ("Partners", PARTNERS_TABLE),
    ("Pipeline Agg (compare)", PIPELINE_AGG_TABLE),
]:
    print(f"  {name}: {table}")

## 1. Current Stores & Sales

In [ ]:
# Load raw stores
stores_raw = spark.table(CURRENT_STORES_TABLE)
print(f"Total stores: {stores_raw.count()}")
print(f"Columns: {stores_raw.columns}")
print(f"\nSales data source:")
display(stores_raw.groupBy("sales_data_source").count())

print("\nStores by state:")
display(stores_raw.groupBy("state").agg(
    F.count("*").alias("count"),
    F.round(F.avg("annual_sales"), 0).alias("avg_sales"),
    F.round(F.min("annual_sales"), 0).alias("min_sales"),
    F.round(F.max("annual_sales"), 0).alias("max_sales")
).orderBy("state"))

In [ ]:
# Sales distribution
stores_pd = stores_raw.select("location_id", "state", "annual_sales").toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(stores_pd['annual_sales'], bins=30, alpha=0.7, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Annual Sales ($)')
axes[0].set_ylabel('Count')
axes[0].set_title('Sales Distribution (All Stores)')
axes[0].axvline(stores_pd['annual_sales'].mean(), color='red', linestyle='--', label=f"Mean: ${stores_pd['annual_sales'].mean():,.0f}")
axes[0].legend()

# Box plot by state
stores_pd.boxplot(column='annual_sales', by='state', ax=axes[1])
axes[1].set_title('Sales by State')
axes[1].set_xlabel('State')
axes[1].set_ylabel('Annual Sales ($)')
plt.suptitle('')

plt.tight_layout()
plt.show()

# Flag issues
null_sales = stores_pd['annual_sales'].isna().sum()
zero_sales = (stores_pd['annual_sales'] == 0).sum()
print(f"\nData quality: {null_sales} null sales, {zero_sales} zero sales")
if null_sales > 0 or zero_sales > 0:
    print("WARNING: Stores with null/zero sales will skew the model!")

## 2. Load Pre-Computed Trade Areas (Isochrones)

In [ ]:
# Load existing store isochrones
isochrones_lce = spark.table(ISOCHRONES_LCE_TABLE)
print(f"Existing store isochrones: {isochrones_lce.count()}")
print(f"Columns: {isochrones_lce.columns}")

print("\nArea (sq km) distribution:")
display(isochrones_lce.select("area_sqkm").summary())

print("\nIsochrones by state:")
display(isochrones_lce.groupBy("state").agg(
    F.count("*").alias("count"),
    F.round(F.avg("area_sqkm"), 2).alias("avg_area_sqkm")
).orderBy("state"))

display(isochrones_lce.limit(3))

In [ ]:
# Load candidate isochrones
candidate_isochrones = spark.table(CANDIDATE_ISO_TABLE)
print(f"Candidate isochrones: {candidate_isochrones.count()}")
print(f"Columns: {candidate_isochrones.columns}")

print("\nArea (sq km) distribution:")
display(candidate_isochrones.select("area_sqkm").summary())

display(candidate_isochrones.limit(3))

## 3. Load H3 Features (CARTO)

In [ ]:
# Load clean H3 features
h3_features = spark.table(H3_FEATURES_TABLE)
print(f"Total H3 cells: {h3_features.count():,}")
print(f"Columns: {h3_features.columns}")

print("\nH3 cells by state:")
display(h3_features.groupBy("state_abbr").agg(
    F.count("*").alias("cell_count"),
    F.round(F.avg("population"), 0).alias("avg_pop"),
    F.round(F.avg("total_poi_count"), 0).alias("avg_poi"),
    F.round(F.avg("human_activity_index"), 2).alias("avg_hai")
).orderBy("state_abbr"))

print("\nFeature distributions:")
display(h3_features.select(
    "population", "target_demographic_total", "total_poi_count",
    "human_activity_index", "retail", "food_drink"
).summary())

## 4. Enrich Trade Areas (Step-by-Step Aggregation)

This is the critical section. We reproduce exactly what `agg_h3_features_current_stores` does, with prints at every step.

### 4a. H3 Polyfill — Index Trade Areas with H3 Cells

In [ ]:
# Standardize isochrone columns (same as pipeline)
iso_columns = isochrones_lce.columns
id_col = next((c for c in iso_columns if c in ['store_number', 'location_id', 'id']), iso_columns[0])

base_ta = isochrones_lce.select(
    col(id_col).alias("store_number"),
    col("latitude"),
    col("longitude"),
    col("store_type"),
    col("city") if "city" in iso_columns else lit(None).alias("city"),
    col("state") if "state" in iso_columns else lit(None).alias("state"),
    col("drive_time_minutes") if "drive_time_minutes" in iso_columns else lit(5).alias("drive_time_minutes"),
    col("area_sqkm") if "area_sqkm" in iso_columns else lit(None).alias("area_sqkm"),
    col("geometry")
)

print(f"Trade areas loaded: {base_ta.count()} stores")
print(f"ID column used: '{id_col}'")
display(base_ta.select("store_number", "state", "area_sqkm").limit(5))

In [ ]:
# H3 Polyfill: find all H3 cells whose centers fall inside each isochrone
ta_h3 = base_ta.withColumn(
    "h3_cell_id",
    explode(expr(f"h3_polyfillash3string(ST_AsBinary(geometry), {H3_RESOLUTION})"))
)

total_h3_cells = ta_h3.count()
print(f"Total H3 cells across all stores: {total_h3_cells:,}")

# Cells per store distribution
cells_per_store = ta_h3.groupBy("store_number", "state").agg(
    F.count("h3_cell_id").alias("h3_count")
)

print("\nH3 cells per store:")
display(cells_per_store.select("h3_count").summary())

print("\nTop 10 stores by H3 cell count:")
display(cells_per_store.orderBy(F.desc("h3_count")).limit(10))

print("\nBottom 10 stores by H3 cell count (potential issues):")
display(cells_per_store.orderBy("h3_count").limit(10))

# Flag stores with very few cells
low_cell_stores = cells_per_store.filter(col("h3_count") < 5).count()
if low_cell_stores > 0:
    print(f"\nWARNING: {low_cell_stores} stores have < 5 H3 cells — very small trade areas!")

### 4b. Inner Join with H3 Features

In [ ]:
# Join polyfilled H3 cells with CARTO features
ta_with_features = ta_h3.join(h3_features, "h3_cell_id", "inner")

matched_cells = ta_with_features.count()
match_rate = 100 * matched_cells / total_h3_cells if total_h3_cells > 0 else 0
print(f"Polyfill cells: {total_h3_cells:,}")
print(f"Matched with H3 features: {matched_cells:,}")
print(f"Match rate: {match_rate:.1f}%")

if match_rate < 90:
    print(f"\nWARNING: {100-match_rate:.1f}% of polyfill cells have NO CARTO data!")
    print("This means some H3 cells inside isochrones are missing from h3_features_clean.")
    print("Possible causes: state boundary gaps, CARTO coverage holes.")

# Check match rate per store
match_by_store = ta_h3.groupBy("store_number").agg(
    F.count("h3_cell_id").alias("polyfill_cells")
).join(
    ta_with_features.groupBy("store_number").agg(
        F.count("h3_cell_id").alias("matched_cells")
    ),
    "store_number",
    "left"
).fillna(0, subset=["matched_cells"]).withColumn(
    "match_pct", F.round(100 * col("matched_cells") / col("polyfill_cells"), 1)
)

print("\nStores with lowest match rates (potential data gaps):")
display(match_by_store.orderBy("match_pct").limit(10))

# Any stores with 0 matches?
zero_match = match_by_store.filter(col("matched_cells") == 0).count()
if zero_match > 0:
    print(f"\nCRITICAL: {zero_match} stores have ZERO matched H3 cells!")

In [ ]:
# Show a sample of the joined data — what does one store look like at H3 level?
sample_store = ta_with_features.select("store_number").first()[0]
print(f"Sample: store {sample_store} — individual H3 cells before aggregation:")
display(
    ta_with_features.filter(col("store_number") == sample_store)
    .select("store_number", "h3_cell_id", "population", "target_demographic_total",
            "retail", "food_drink", "human_activity_index", "urbanity")
    .limit(20)
)

### 4c. Aggregate by Store

Aggregation rules:
- **SUM**: population, target_demographic_total, all 8 POI categories, total_poi_count
- **AVG**: human_activity_index (CARTO 0-100 normalized score)
- **COUNT**: h3_cell_count (trade area size proxy)
- **FIRST**: urbanity, geometry

In [ ]:
# Define aggregation columns
poi_cols = ['retail', 'food_drink', 'leisure', 'education', 'healthcare', 'financial', 'tourism', 'transportation']
demo_cols = ['population', 'target_demographic_total']

# Check which columns actually exist
available_cols = ta_with_features.columns
existing_poi = [c for c in poi_cols if c in available_cols]
existing_demo = [c for c in demo_cols if c in available_cols]

print(f"POI columns found: {existing_poi}")
print(f"Demo columns found: {existing_demo}")
print(f"human_activity_index: {'human_activity_index' in available_cols}")
print(f"urbanity: {'urbanity' in available_cols}")
print(f"total_poi_count: {'total_poi_count' in available_cols}")

# Build aggregation expressions
agg_exprs = []

# SUM demographics
for c in existing_demo:
    agg_exprs.append(F.sum(F.coalesce(col(c), lit(0))).cast("long").alias(c))

# SUM POI counts
for c in existing_poi:
    agg_exprs.append(F.sum(F.coalesce(col(c), lit(0))).cast("long").alias(c))

# SUM total_poi_count
if 'total_poi_count' in available_cols:
    agg_exprs.append(F.sum(F.coalesce(col("total_poi_count"), lit(0))).cast("long").alias("total_poi_count"))

# AVG human_activity_index
if 'human_activity_index' in available_cols:
    agg_exprs.append(F.avg(F.coalesce(col("human_activity_index"), lit(0))).alias("human_activity_index"))

# FIRST urbanity
if 'urbanity' in available_cols:
    agg_exprs.append(F.first(col("urbanity")).alias("urbanity"))

# COUNT + FIRST geometry
agg_exprs.append(F.count("h3_cell_id").alias("h3_cell_count"))
agg_exprs.append(F.first("geometry").alias("geometry"))

# Aggregate
store_features_agg = ta_with_features.groupBy(
    "store_number", "latitude", "longitude", "store_type",
    "city", "state", "drive_time_minutes", "area_sqkm"
).agg(*agg_exprs)

print(f"\nAggregated features for {store_features_agg.count()} stores")
print(f"Columns: {store_features_agg.columns}")
display(store_features_agg.limit(5))

In [ ]:
# Join sales data
current_stores = spark.table(CURRENT_STORES_TABLE).select("location_id", "annual_sales", "monthly_sales")

store_features_with_sales = store_features_agg.join(
    current_stores,
    store_features_agg["store_number"] == current_stores["location_id"],
    "left"
).drop("location_id")

print(f"Stores with sales data: {store_features_with_sales.filter(col('annual_sales').isNotNull()).count()}")
print(f"Stores missing sales: {store_features_with_sales.filter(col('annual_sales').isNull()).count()}")

# Show features alongside sales — this is the key view for troubleshooting
print("\nAggregated features with sales (sample):")
display(
    store_features_with_sales
    .select("store_number", "state", "annual_sales", "population", "target_demographic_total",
            "h3_cell_count", "area_sqkm", "human_activity_index",
            "retail", "food_drink", "total_poi_count")
    .orderBy(F.desc("annual_sales"))
    .limit(20)
)

In [ ]:
# Summary statistics of aggregated features
print("Aggregated feature distributions:")
display(
    store_features_with_sales.select(
        "annual_sales", "population", "target_demographic_total",
        "h3_cell_count", "area_sqkm", "human_activity_index",
        "retail", "food_drink", "total_poi_count"
    ).summary()
)

print("\nBy state:")
display(
    store_features_with_sales.groupBy("state").agg(
        F.count("*").alias("stores"),
        F.round(F.avg("annual_sales"), 0).alias("avg_sales"),
        F.round(F.avg("population"), 0).alias("avg_pop"),
        F.round(F.avg("h3_cell_count"), 0).alias("avg_h3_cells"),
        F.round(F.avg("area_sqkm"), 2).alias("avg_area_sqkm"),
        F.round(F.avg("human_activity_index"), 2).alias("avg_hai")
    ).orderBy("state")
)

### 4d. Folium Map — Visualize H3 Cells per Store

Pick a few sample stores and render their isochrone + H3 cells to visually verify polyfill.

In [ ]:
import folium
import h3

# Pick 3 sample stores: one with many H3 cells, one median, one with few
cell_counts = cells_per_store.orderBy("h3_count").toPandas()
sample_stores = [
    cell_counts.iloc[0]['store_number'],              # fewest cells
    cell_counts.iloc[len(cell_counts)//2]['store_number'],  # median
    cell_counts.iloc[-1]['store_number'],              # most cells
]
print(f"Sample stores for map: {sample_stores}")

# Collect data for these stores
sample_ta = base_ta.filter(col("store_number").isin(sample_stores)).toPandas()
sample_h3 = ta_h3.filter(col("store_number").isin(sample_stores)).select(
    "store_number", "h3_cell_id"
).toPandas()

# Create map centered on first sample store
center_lat = sample_ta['latitude'].mean()
center_lon = sample_ta['longitude'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=10)

colors = ['blue', 'green', 'purple']

for i, store_id in enumerate(sample_stores):
    store_row = sample_ta[sample_ta['store_number'] == store_id].iloc[0]
    store_h3_cells = sample_h3[sample_h3['store_number'] == store_id]['h3_cell_id'].tolist()
    color = colors[i % len(colors)]
    
    # Store marker
    folium.CircleMarker(
        location=[store_row['latitude'], store_row['longitude']],
        radius=8, color='red', fillColor='red', fillOpacity=0.8,
        popup=f"Store {store_id} ({len(store_h3_cells)} H3 cells)"
    ).add_to(m)
    
    # H3 hexagons
    for cell_id in store_h3_cells:
        try:
            boundary = h3.cell_to_boundary(cell_id)
            coords = [[lat, lon] for lat, lon in boundary]
            folium.Polygon(
                locations=coords, color=color, fillColor=color,
                fillOpacity=0.15, weight=1
            ).add_to(m)
        except:
            pass

print(f"Map shows {len(sample_stores)} stores with their H3 polyfill cells")
print("Red markers = store locations, colored hexagons = H3 cells in trade area")
m

### 4e. Competitor & Partner Counts

In [ ]:
# Count pizza competitors and partner stores within each store's trade area via H3 join
try:
    competitors_h3 = spark.table(COMPETITORS_TABLE).select(
        "poi_id",
        expr("h3_longlatash3string(longitude, latitude, 8)").alias("h3_cell_id")
    )
    partners_h3 = spark.table(PARTNERS_TABLE).select(
        "poi_id",
        expr("h3_longlatash3string(longitude, latitude, 8)").alias("h3_cell_id")
    )
    
    print(f"Competitor POIs: {competitors_h3.count()}")
    print(f"Partner POIs: {partners_h3.count()}")

    # Count distinct competitors per store's trade area
    competitor_counts = ta_h3.join(competitors_h3, "h3_cell_id", "inner") \
        .groupBy("store_number").agg(F.countDistinct("poi_id").alias("competitor_count"))

    partner_counts = ta_h3.join(partners_h3, "h3_cell_id", "inner") \
        .groupBy("store_number").agg(F.countDistinct("poi_id").alias("partner_count"))

    # Join to features
    store_features_with_sales = store_features_with_sales \
        .join(competitor_counts, "store_number", "left") \
        .join(partner_counts, "store_number", "left") \
        .fillna(0, subset=["competitor_count", "partner_count"])

    print("\nCompetitor/partner count distribution:")
    display(store_features_with_sales.select("competitor_count", "partner_count").summary())
    
    print("\nSample:")
    display(
        store_features_with_sales
        .select("store_number", "state", "annual_sales", "competitor_count", "partner_count")
        .orderBy(F.desc("competitor_count")).limit(10)
    )
    
except Exception as e:
    print(f"Could not load competitor/partner tables: {e}")
    store_features_with_sales = store_features_with_sales \
        .withColumn("competitor_count", lit(0)) \
        .withColumn("partner_count", lit(0))

## 5. Compare Against Pipeline Output

Side-by-side comparison of our step-by-step aggregation vs the pipeline's `current_stores_features_agg`.

In [ ]:
# Load pipeline output
try:
    pipeline_agg = spark.table(PIPELINE_AGG_TABLE)
    print(f"Pipeline output: {pipeline_agg.count()} stores")
    print(f"Pipeline columns: {pipeline_agg.columns}")
    
    # Compare key metrics for first 10 stores
    compare_cols = ["population", "target_demographic_total", "h3_cell_count", 
                    "human_activity_index", "total_poi_count", "annual_sales"]
    
    ours = store_features_with_sales.select(
        col("store_number"),
        *[col(c).alias(f"ours_{c}") for c in compare_cols if c in store_features_with_sales.columns]
    )
    
    theirs = pipeline_agg.select(
        col("store_number"),
        *[col(c).alias(f"pipeline_{c}") for c in compare_cols if c in pipeline_agg.columns]
    )
    
    comparison = ours.join(theirs, "store_number", "inner")
    
    print(f"\nMatched {comparison.count()} stores for comparison")
    print("\nSide-by-side (first 10 stores):")
    display(comparison.limit(10))
    
    # Check for discrepancies in population
    if "ours_population" in comparison.columns and "pipeline_population" in comparison.columns:
        discrepancies = comparison.withColumn(
            "pop_diff", F.abs(col("ours_population") - col("pipeline_population"))
        ).filter(col("pop_diff") > 0)
        
        if discrepancies.count() > 0:
            print(f"\nWARNING: {discrepancies.count()} stores have population differences!")
            display(discrepancies.orderBy(F.desc("pop_diff")).limit(5))
        else:
            print("\nPopulation values match exactly.")
    
except Exception as e:
    print(f"Could not load pipeline output for comparison: {e}")
    print("Run the pipeline first, or skip this comparison.")

## 6. Prediction Model Exploration

### 6a. Feature Correlations

In [ ]:
# Define feature columns (same as pipeline)
feature_columns = [
    'population', 'target_demographic_total',
    'retail', 'food_drink', 'leisure', 'education',
    'healthcare', 'financial', 'tourism', 'transportation',
    'competitor_count', 'partner_count',
    'human_activity_index',
    'h3_cell_count', 'area_sqkm',
]
target_column = 'annual_sales'

# Filter to only columns that exist in our DataFrame
available_features = [c for c in feature_columns if c in store_features_with_sales.columns]
missing_features = [c for c in feature_columns if c not in store_features_with_sales.columns]
if missing_features:
    print(f"WARNING: Missing features: {missing_features}")

print(f"Using {len(available_features)} features: {available_features}")

# Convert to Pandas
train_pd = store_features_with_sales.select(
    available_features + [target_column, 'store_number', 'state']
).toPandas()

# Drop rows with null sales
train_pd = train_pd.dropna(subset=[target_column])
train_pd = train_pd[train_pd[target_column] > 0]
print(f"Training samples: {len(train_pd)}")

# Normalize state names
state_mapping = {
    'Massachusetts': 'MA', 'michigan': 'MI', 'Michigan': 'MI',
    'Virginia': 'VA', 'New York': 'NY', 'Washington': 'WA',
    'Maryland': 'MD', 'New Jersey': 'NJ',
    'MA': 'MA', 'MI': 'MI', 'VA': 'VA', 'NY': 'NY',
    'WA': 'WA', 'MD': 'MD', 'NJ': 'NJ',
}
train_pd['state'] = train_pd['state'].map(lambda x: state_mapping.get(x, x))
print(f"\nStores by state:\n{train_pd['state'].value_counts().sort_index()}")

In [ ]:
# Correlation matrix
X = train_pd[available_features]
y = train_pd[target_column]

corr_with_target = X.corrwith(y).sort_values(ascending=False)
print("Feature correlations with annual_sales:")
print(corr_with_target)

# Heatmap
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Full correlation matrix
corr_matrix = X.join(y).corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, ax=axes[0], cbar_kws={"shrink": 0.8})
axes[0].set_title('Feature Correlation Matrix', fontsize=12)

# Bar chart of correlations with sales
corr_with_target.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_xlabel('Correlation with Annual Sales')
axes[1].set_title('Feature Correlations with Sales', fontsize=12)
axes[1].axvline(x=0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

# Highly correlated feature pairs
print("\nHighly correlated feature pairs (|r| > 0.85):")
for i in range(len(X.columns)):
    for j in range(i+1, len(X.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.85:
            print(f"  {X.columns[i]} <-> {X.columns[j]}: {r:.3f}")

### 6b. Normalize Features by Population

Create per-capita versions to see if they correlate better with sales.

In [ ]:
# Create per-capita features
per_capita_features = []
train_norm = train_pd.copy()

normalize_cols = ['retail', 'food_drink', 'leisure', 'education', 'healthcare',
                  'financial', 'tourism', 'transportation', 'target_demographic_total']
normalize_cols = [c for c in normalize_cols if c in available_features]

for feat in normalize_cols:
    new_name = f"{feat}_per_capita"
    train_norm[new_name] = train_norm[feat] / train_norm['population'].replace(0, np.nan)
    per_capita_features.append(new_name)

# Also add sales per capita for reference
train_norm['sales_per_capita'] = train_norm['annual_sales'] / train_norm['population'].replace(0, np.nan)

print("Per-capita feature correlations with annual_sales:")
pc_corr = train_norm[per_capita_features].corrwith(train_norm['annual_sales']).sort_values(ascending=False)
print(pc_corr)

print("\nOriginal vs per-capita correlations:")
for feat in normalize_cols:
    orig_r = corr_with_target.get(feat, 0)
    pc_r = pc_corr.get(f"{feat}_per_capita", 0)
    better = "*" if abs(pc_r) > abs(orig_r) else ""
    print(f"  {feat:30s}  original: {orig_r:+.3f}  per_capita: {pc_r:+.3f} {better}")

# Scatter plots: per-capita features vs sales
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
plot_feats = per_capita_features[:6]
for i, feat in enumerate(plot_feats):
    ax = axes[i // 3][i % 3]
    ax.scatter(train_norm[feat], train_norm['annual_sales'], alpha=0.3, s=15)
    ax.set_xlabel(feat)
    ax.set_ylabel('Annual Sales')
    r = train_norm[feat].corr(train_norm['annual_sales'])
    ax.set_title(f'r = {r:.3f}')
plt.suptitle('Per-Capita Features vs Sales', fontsize=14)
plt.tight_layout()
plt.show()

### 6c. Sales Distribution Analysis

Explore whether log transform is needed and what alternatives exist.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Raw sales
axes[0].hist(y, bins=30, alpha=0.7, color='steelblue', edgecolor='white')
axes[0].set_title(f'Raw Sales (skew={y.skew():.2f})')
axes[0].set_xlabel('Annual Sales ($)')

# Log-transformed
y_log = np.log1p(y)
axes[1].hist(y_log, bins=30, alpha=0.7, color='coral', edgecolor='white')
axes[1].set_title(f'Log(1+Sales) (skew={y_log.skew():.2f})')
axes[1].set_xlabel('Log(1 + Annual Sales)')

# Sqrt-transformed
y_sqrt = np.sqrt(y)
axes[2].hist(y_sqrt, bins=30, alpha=0.7, color='mediumseagreen', edgecolor='white')
axes[2].set_title(f'Sqrt(Sales) (skew={y_sqrt.skew():.2f})')
axes[2].set_xlabel('Sqrt(Annual Sales)')

plt.suptitle('Target Variable Distributions', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Raw sales: mean=${y.mean():,.0f}, std=${y.std():,.0f}, CV={y.std()/y.mean():.2f}")
print(f"\nNote: XGBoost does NOT need the target scaled to 0-1. It's tree-based —")
print(f"splits are rank-based. Log transform helps when residual variance scales with target.")
print(f"We'll test both raw and log-transformed targets below.")

### 6d. Model Comparison

Train 3 model types x 2 target transforms = 6 variants, all with stratified 5-fold CV.

In [ ]:
# Setup MLflow — use a SEPARATE experiment from the pipeline
experiment_name = f"/Users/{spark.sql('SELECT current_user()').collect()[0][0]}/geospatial-retail-standalone"
mlflow.set_experiment(experiment_name)
mlflow.end_run()  # close any orphaned runs
print(f"MLflow experiment: {experiment_name}")

# Prepare data
X_train = train_pd[available_features].copy()
y_raw = train_pd[target_column].copy()
y_log = np.log1p(y_raw)
state_labels = train_pd['state']

print(f"Training data: {X_train.shape[0]} stores, {X_train.shape[1]} features")
print(f"States: {state_labels.nunique()} ({state_labels.value_counts().to_dict()})")

In [ ]:
# Model definitions
model_configs = {
    "LinearRegression_raw": {
        "model": LinearRegression(),
        "target": "raw",
        "label": "Linear Regression (raw target)"
    },
    "LinearRegression_log": {
        "model": LinearRegression(),
        "target": "log",
        "label": "Linear Regression (log target)"
    },
    "XGBoost_raw": {
        "model": xgb.XGBRegressor(
            n_estimators=100, max_depth=3, min_child_weight=5,
            learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=1.0, random_state=42, n_jobs=-1
        ),
        "target": "raw",
        "label": "XGBoost (raw target)"
    },
    "XGBoost_log": {
        "model": xgb.XGBRegressor(
            n_estimators=100, max_depth=3, min_child_weight=5,
            learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=1.0, random_state=42, n_jobs=-1
        ),
        "target": "log",
        "label": "XGBoost (log target) [current pipeline]"
    },
    "RandomForest_raw": {
        "model": RandomForestRegressor(
            n_estimators=100, max_depth=10, min_samples_leaf=5,
            random_state=42, n_jobs=-1
        ),
        "target": "raw",
        "label": "Random Forest (raw target)"
    },
    "RandomForest_log": {
        "model": RandomForestRegressor(
            n_estimators=100, max_depth=10, min_samples_leaf=5,
            random_state=42, n_jobs=-1
        ),
        "target": "log",
        "label": "Random Forest (log target)"
    },
}

print(f"Will train {len(model_configs)} model variants with stratified 5-fold CV")
for name, cfg in model_configs.items():
    print(f"  - {cfg['label']}")

In [ ]:
# Run stratified 5-fold CV for all model variants
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

all_results = {}
all_oof_preds = {}

parent_run = mlflow.start_run(run_name="model_comparison")

for model_name, config in model_configs.items():
    print(f"\n{'='*60}")
    print(f"Training: {config['label']}")
    print(f"{'='*60}")
    
    y_target = y_log if config['target'] == 'log' else y_raw
    
    cv_results = []
    oof_preds = np.zeros(len(y_raw))
    
    with mlflow.start_run(run_name=model_name, nested=True):
        for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X_train, state_labels)):
            X_tr = X_train.iloc[train_idx]
            X_val = X_train.iloc[val_idx]
            y_tr = y_target.iloc[train_idx]
            y_val_actual = y_raw.iloc[val_idx]  # always evaluate on raw scale
            
            # Clone model for each fold
            from sklearn.base import clone
            fold_model = clone(config['model'])
            fold_model.fit(X_tr, y_tr)
            
            # Predict
            preds = fold_model.predict(X_val)
            if config['target'] == 'log':
                preds = np.expm1(preds)  # back-transform
            preds = np.maximum(preds, 0)  # floor at 0
            
            oof_preds[val_idx] = preds
            
            rmse = np.sqrt(mean_squared_error(y_val_actual, preds))
            mae = mean_absolute_error(y_val_actual, preds)
            r2 = r2_score(y_val_actual, preds)
            mape = mean_absolute_percentage_error(y_val_actual, preds)
            
            cv_results.append({'fold': fold_idx+1, 'rmse': rmse, 'mae': mae, 'r2': r2, 'mape': mape})
            print(f"  Fold {fold_idx+1}: RMSE=${rmse:,.0f}, MAE=${mae:,.0f}, R2={r2:.3f}, MAPE={mape:.1%}")
        
        cv_df = pd.DataFrame(cv_results)
        
        # Log to MLflow
        mlflow.log_param("model_type", model_name.split('_')[0])
        mlflow.log_param("target_transform", config['target'])
        mlflow.log_param("n_features", len(available_features))
        mlflow.log_param("features", ",".join(available_features))
        mlflow.log_metric("mean_cv_rmse", cv_df['rmse'].mean())
        mlflow.log_metric("mean_cv_mae", cv_df['mae'].mean())
        mlflow.log_metric("mean_cv_r2", cv_df['r2'].mean())
        mlflow.log_metric("mean_cv_mape", cv_df['mape'].mean())
        mlflow.log_metric("std_cv_r2", cv_df['r2'].std())
        
        print(f"\n  MEAN: RMSE=${cv_df['rmse'].mean():,.0f}, R2={cv_df['r2'].mean():.3f} (+/-{cv_df['r2'].std():.3f}), MAPE={cv_df['mape'].mean():.1%}")
    
    all_results[model_name] = cv_df
    all_oof_preds[model_name] = oof_preds

mlflow.end_run()
print(f"\nAll models logged to MLflow experiment: {experiment_name}")

In [ ]:
# Summary comparison table
summary_rows = []
for name, cv_df in all_results.items():
    cfg = model_configs[name]
    summary_rows.append({
        'Model': cfg['label'],
        'Mean RMSE': f"${cv_df['rmse'].mean():,.0f}",
        'Mean MAE': f"${cv_df['mae'].mean():,.0f}",
        'Mean R2': f"{cv_df['r2'].mean():.3f}",
        'R2 Std': f"{cv_df['r2'].std():.3f}",
        'Mean MAPE': f"{cv_df['mape'].mean():.1%}",
        'RMSE_numeric': cv_df['rmse'].mean(),  # for sorting
    })

summary_df = pd.DataFrame(summary_rows).sort_values('RMSE_numeric')
print("\n" + "="*80)
print("MODEL COMPARISON SUMMARY (sorted by RMSE)")
print("="*80)
display(summary_df.drop(columns=['RMSE_numeric']))

In [ ]:
# Out-of-fold prediction plots for all models
n_models = len(all_oof_preds)
cols = 3
rows = (n_models + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 5*rows))
axes = axes.flatten() if n_models > 1 else [axes]

for i, (name, preds) in enumerate(all_oof_preds.items()):
    ax = axes[i]
    cfg = model_configs[name]
    cv_df = all_results[name]
    
    ax.scatter(y_raw, preds, alpha=0.3, s=15)
    ax.plot([y_raw.min(), y_raw.max()], [y_raw.min(), y_raw.max()], 'r--', alpha=0.7)
    ax.set_xlabel('Actual Sales ($)')
    ax.set_ylabel('Predicted Sales ($)')
    ax.set_title(f"{cfg['label']}\nR2={cv_df['r2'].mean():.3f}")

# Hide unused axes
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Out-of-Fold Predictions: Actual vs Predicted', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 6e. Feature Importance (Best Model)

In [ ]:
# Train final models on all data for feature importance comparison
# Use the best-performing target transform for each model type

best_model_name = min(all_results.keys(), key=lambda k: all_results[k]['rmse'].mean())
best_config = model_configs[best_model_name]
print(f"Best model: {best_config['label']}")
print(f"Training on all {len(X_train)} stores for feature importance...")

y_final = y_log if best_config['target'] == 'log' else y_raw
from sklearn.base import clone
final_model = clone(best_config['model'])
final_model.fit(X_train, y_final)

# Feature importance — works for tree-based models
if hasattr(final_model, 'feature_importances_'):
    importance_df = pd.DataFrame({
        'feature': available_features,
        'importance': final_model.feature_importances_
    }).sort_values('importance', ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    importance_df.plot(x='feature', y='importance', kind='barh', ax=ax, color='steelblue', legend=False)
    ax.set_xlabel('Feature Importance')
    ax.set_title(f'Feature Importance: {best_config["label"]}')
    plt.tight_layout()
    plt.show()

elif hasattr(final_model, 'coef_'):
    coef_df = pd.DataFrame({
        'feature': available_features,
        'coefficient': final_model.coef_
    }).sort_values('coefficient', ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    coef_df.plot(x='feature', y='coefficient', kind='barh', ax=ax, color='steelblue', legend=False)
    ax.set_xlabel('Coefficient')
    ax.set_title(f'Coefficients: {best_config["label"]}')
    ax.axvline(x=0, color='black', linewidth=0.5)
    plt.tight_layout()
    plt.show()

## 7. Candidate Predictions (MA)

Apply the best model to candidates and compare with pipeline output.

In [ ]:
# Load candidate aggregated features from pipeline
try:
    candidates_agg = spark.table(PIPELINE_CAND_AGG)
    print(f"Loaded {candidates_agg.count():,} candidates from {PIPELINE_CAND_AGG}")
    
    # Prepare candidate features
    cand_pd = candidates_agg.select(
        ['candidate_id'] + [c for c in available_features if c in candidates_agg.columns]
    ).toPandas()
    
    # Fill missing columns
    for c in available_features:
        if c not in cand_pd.columns:
            cand_pd[c] = 0
            print(f"  WARNING: '{c}' missing from candidates, filled with 0")
    
    X_cand = cand_pd[available_features]
    
    # Predict
    preds = final_model.predict(X_cand)
    if best_config['target'] == 'log':
        preds = np.expm1(preds)
    preds = np.maximum(preds, 0)
    
    cand_pd['predicted_annual_sales'] = preds.astype(int)
    
    print(f"\nPrediction Summary (our model - {best_config['label']}):")
    print(f"  Mean:   ${cand_pd['predicted_annual_sales'].mean():,.0f}")
    print(f"  Median: ${cand_pd['predicted_annual_sales'].median():,.0f}")
    print(f"  Min:    ${cand_pd['predicted_annual_sales'].min():,.0f}")
    print(f"  Max:    ${cand_pd['predicted_annual_sales'].max():,.0f}")
    
    # Compare with pipeline predictions
    try:
        pipeline_preds = spark.table(PIPELINE_CAND_FINAL).select(
            "candidate_id",
            col("predicted_annual_sales").alias("pipeline_predicted_sales")
        ).toPandas()
        
        comparison = cand_pd[['candidate_id', 'predicted_annual_sales']].merge(
            pipeline_preds, on='candidate_id', how='inner'
        )
        comparison['diff'] = comparison['predicted_annual_sales'] - comparison['pipeline_predicted_sales']
        comparison['diff_pct'] = 100 * comparison['diff'] / comparison['pipeline_predicted_sales']
        
        print(f"\nComparison with pipeline predictions ({len(comparison)} candidates):")
        print(f"  Pipeline mean: ${comparison['pipeline_predicted_sales'].mean():,.0f}")
        print(f"  Our mean:      ${comparison['predicted_annual_sales'].mean():,.0f}")
        print(f"  Avg diff:      ${comparison['diff'].mean():,.0f} ({comparison['diff_pct'].mean():.1f}%)")
        print(f"  Correlation:   {comparison['predicted_annual_sales'].corr(comparison['pipeline_predicted_sales']):.3f}")
        
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.scatter(comparison['pipeline_predicted_sales'], comparison['predicted_annual_sales'], alpha=0.3, s=10)
        lims = [min(comparison['pipeline_predicted_sales'].min(), comparison['predicted_annual_sales'].min()),
                max(comparison['pipeline_predicted_sales'].max(), comparison['predicted_annual_sales'].max())]
        ax.plot(lims, lims, 'r--', alpha=0.7)
        ax.set_xlabel('Pipeline Predictions ($)')
        ax.set_ylabel(f'Our Predictions ({best_config["label"]}) ($)')
        ax.set_title('Pipeline vs Exploration Predictions')
        plt.tight_layout()
        plt.show()
        
    except Exception as e:
        print(f"Could not compare with pipeline predictions: {e}")

except Exception as e:
    print(f"Could not load candidates: {e}")
    print("Run the pipeline's agg_h3_features_candidates first.")

In [ ]:
# Distribution of predictions
try:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(cand_pd['predicted_annual_sales'], bins=30, alpha=0.7, color='steelblue', edgecolor='white')
    axes[0].axvline(y_raw.mean(), color='red', linestyle='--', label=f'Training mean: ${y_raw.mean():,.0f}')
    axes[0].set_xlabel('Predicted Annual Sales ($)')
    axes[0].set_ylabel('Count')
    axes[0].set_title(f'Candidate Predictions ({best_config["label"]})')
    axes[0].legend()
    
    # Training vs prediction distributions
    axes[1].hist(y_raw, bins=30, alpha=0.5, color='steelblue', edgecolor='white', label='Training (actual)')
    axes[1].hist(cand_pd['predicted_annual_sales'], bins=30, alpha=0.5, color='coral', edgecolor='white', label='Candidate (predicted)')
    axes[1].set_xlabel('Annual Sales ($)')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Training Actuals vs Candidate Predictions')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
except:
    pass

## Summary

Key findings from this exploration:

In [ ]:
print("=" * 70)
print("PIPELINE TROUBLESHOOTING SUMMARY")
print("=" * 70)

print(f"\n1. DATA:")
print(f"   Stores: {len(train_pd)} across {train_pd['state'].nunique()} states")
print(f"   Features: {len(available_features)}")
print(f"   H3 match rate: {match_rate:.1f}%")

print(f"\n2. AGGREGATION VERIFICATION:")
print(f"   H3 polyfill cells per store: {total_h3_cells/len(train_pd):.0f} avg")
print(f"   Stores with < 5 cells: {low_cell_stores}")
print(f"   Stores with 0 matched cells: {zero_match}")

print(f"\n3. MODEL COMPARISON:")
for name, cv_df in sorted(all_results.items(), key=lambda x: x[1]['rmse'].mean()):
    cfg = model_configs[name]
    print(f"   {cfg['label']:45s} RMSE=${cv_df['rmse'].mean():>8,.0f}  R2={cv_df['r2'].mean():>6.3f}  MAPE={cv_df['mape'].mean():.1%}")

best_r2 = all_results[best_model_name]['r2'].mean()
print(f"\n4. BEST MODEL: {model_configs[best_model_name]['label']}")
print(f"   R2={best_r2:.3f}")

if best_r2 < 0:
    print(f"\n   NOTE: Negative R2 means the model predicts worse than the mean.")
    print(f"   Spatial features alone may not be sufficient to predict exact dollar sales.")
    print(f"   Operational factors (management, foot traffic, marketing) likely dominate.")
    print(f"   The model may still produce useful RELATIVE rankings even with low R2.")

print(f"\n5. CHECK MLflow experiment for full comparison:")
print(f"   {experiment_name}")

print("\n" + "=" * 70)